In [11]:
  import pandas as pd
  print(pd.__version__)

3.0.6


In [12]:
STOPS_PATH = "../data/raw/itm_north_west_gtfs/stops.txt"
df = pd.read_csv(STOPS_PATH,dtype={"stop_id": str, "stop_code": str})
df.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,wheelchair_boarding,location_type,parent_station,platform_code
0,2500DCL792,landmwdg,New Line,53.690549,-2.182164,0,0,NaN,NaN
1,090033104077,cumdpjmt,Ness Way,54.884603,-2.968188,0,0,NaN,NaN
2,0600MA19191,cheagjap,Smith Lane,53.331856,-2.339258,0,0,NaN,NaN
3,2800S44079B,merdmpjg,Eaton Road,53.430030,-2.909250,0,0,NaN,NaN
4,2500IMG2876,langjdpa,Rosehill Road,53.782133,-2.253631,0,0,NaN,NaN


In [13]:
df.shape


(36785, 9)

In [14]:
df.isnull().sum()

stop_id                    0
stop_code               1941
stop_name                  0
stop_lat                   0
stop_lon                   0
wheelchair_boarding        0
location_type              0
parent_station         36082
platform_code          36255
dtype: int64

In [15]:
df.describe()

,stop_lat,stop_lon,wheelchair_boarding,location_type
count,36785.000000,36785.000000,36785.0,36785.00000
mean,53.624556,-2.583132,0.0,0.00560
std,0.413776,0.370889,0.0,0.07535
min,51.156573,-4.251747,0.0,0.00000
25%,53.403710,-2.900430,0.0,0.00000
50%,53.514449,-2.576561,0.0,0.00000
75%,53.733254,-2.255969,0.0,0.00000
max,55.955515,-0.060961,0.0,2.00000


In [16]:
min_lat , max_lat = 53.33, 53.69
min_lon, max_lon = -2.73, -1.91
gm_stops = df[
    df["stop_lat"].between(min_lat,max_lat)
    & df["stop_lon"].between(min_lon,max_lon)
].copy()
print("stops in Greater Manchester:",len(gm_stops))
print("stops removed:", len(df)- len(gm_stops))
gm_stops["stop_name"].sample(10 , random_state=42)

stops in Greater Manchester: 15522
stops removed: 21263


7610                                Belle Vue
24171                             Queens Road
17098                              Ewart Road
35818                            Clifton Road
8098                             Rutland Road
244      East Didsbury (Manchester Metrolink)
33521                         Dearnley Avenue
12909                         Mancroft Avenue
23336                          Osborne Street
30376                     St Augustine Street
Name: stop_name, dtype: str

In [17]:
gm_stops.to_parquet("../data/processed/gm_stops.parquet", index=False)

In [18]:
check = pd.read_parquet("../data/processed/gm_stops.parquet")
print(check.shape)

(15522, 9)


In [19]:
STOP_TIMES_PATH = "../data/raw/itm_north_west_gtfs/stop_times.txt"

stop_times = pd.read_csv(
    STOP_TIMES_PATH,
    usecols=["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    dtype={"trip_id": str, "stop_id": str},
)

In [20]:
print("All North west Rows:", len(stop_times))

All North west Rows: 5133578


In [21]:
gm_stop_times = stop_times[stop_times["stop_id"].isin(gm_stops["stop_id"])].copy()
print("Greater Manchester rows:", len(gm_stop_times))

Greater Manchester rows: 2761590


In [22]:
gm_stop_times.to_parquet("../data/processed/gm_stop_times.parquet", index=False)

check = pd.read_parquet("../data/processed/gm_stop_times.parquet")
print(check.shape)


(2761590, 5)
